## Test if measuring morality could ccorrelate with anthropomorphism
- pay attention to which parts of the sentence are used during the classification process
- do so with XAI methods of choice

In [1]:
import json
from pprint import pprint
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

import torch
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

import spacy
from spacy.matcher import DependencyMatcher
from spacy import displacy

nlp = spacy.load('de_core_news_lg') # oder de_core_news_sm

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using mps device


In [2]:
# load docs (KI sentences) to analyse
with open("generated_data/Leipzig_KI_sentences_all.json", "r") as f:
    leipzig_data = json.load(f)
taz24_data = pd.read_csv("generated_data/KI_sentences_translated.csv")
print("leipzig:", len(leipzig_data),list(leipzig_data.keys()), "\ntaz24:", taz24_data.columns)

leipzig: 31 ['2003_1M', '2013_1M', '2007_1M', '1996_1M', '2017_1M', '2023_1M', '2006_1M', '1997_1M', '2016_1M', '2022_1M', '2002_1M', '2012_1M', '2009_1M', '2019_1M', '1998_1M', '2008_1M', '2018_1M', '1999_1M', '2005_1M', '2015_1M', '2021_1M', '2001_1M', '2025_1M', '2011_1M', '2000_1M', '2010_1M', '2004_1M', '1995_1M', '2014_1M', '2020_1M', '2024'] 
taz24: Index(['id', 'original', 'translation'], dtype='object')


In [3]:
# load docs (anthropomorphic KI sentences) to analyse
with open("generated_data/all_anthro_ki.json", "r") as f:
    anthro_data = json.load(f)
print("anthro:",list(anthro_data.keys()), "len sb:",len(anthro_data["sb"]), "len ob:",len(anthro_data["ob"]), "len v:",len(anthro_data["v"]))

anthro: ['sb', 'ob', 'v'] len sb: 115 len ob: 29 len v: 0


In [4]:
# measure using library?


In [5]:
# ask llm to generate RoT for each doc
# then analzse sb relationships for attention (to find actor (agent))

In [6]:
def get_sent_dep(dep_dict):
    sent = dep_dict['sentence']
    dep = dep_dict['dependent']
    return sent, dep

def get_sent_dep_txt(sentence):
    print("sentence:", sentence)
    doc = nlp(sentence)
    relation = {}
    if doc.has_annotation("DEP"):
        for token in doc:
            print("token:", token.text, "pos:", token.pos_, "dep:", token.dep_)
            if token.pos_ == "VERB":
                for child in token.children:
                    if child.dep_ in ["sb"]:
                        relation = {
                            "type": "verb_subject",
                            "verb": token.text,
                            "dependent": child.text,
                            "dep_label": child.dep_,
                            "verb_pos": token.pos_,
                            "dep_pos": child.pos_,
                            "sentence": doc.text
                        }
    sent, dep = get_sent_dep(relation)
    return relation, sent, dep

In [11]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-german-dbmdz-uncased")
model = AutoModelForCausalLM.from_pretrained("google-bert/bert-base-german-dbmdz-uncased", is_decoder=True)

In [12]:
# prompt bert to gnerate RoT for each sentence in leipzig data
model.eval()
rot_sentences = {}

for id, sentence in tqdm(leipzig_data["2025_1M"].items(), desc="generating RoT"):
    text = sentence.strip()
    #prompt = f"{text} Weil {tokenizer.mask_token} {tokenizer.mask_token} {tokenizer.mask_token}."
    prompt = f"""Definiere Faustregeln in der Form 'Es ist gut/schlecht, wenn ___ , weil ___.'
    Text:{text}
    Faustregel: """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
        )
    rot_sentences[id] = tokenizer.decode(outputs[0], skip_special_tokens=True)


generating RoT: 100%|██████████| 3357/3357 [2:55:38<00:00,  3.14s/it]  


In [13]:
pprint(rot_sentences)

{'100038': "definiere faustregeln in der form ' es ist gut / schlecht, wenn _ "
           "_ _, weil _ _ _. ' text : bei einer simplen mathematischen analyse "
           'der pixel verraten sich bilder die die ki erzeugt hat durch '
           'auffallige muster faustregel '
           ':........................................................................................................................................................................................................',
 '100101': "definiere faustregeln in der form ' es ist gut / schlecht, wenn _ "
           "_ _, weil _ _ _. ' text : bei einer veranstaltung im kreis greiz "
           'stand das thema kunstliche intelligenz im mittelpunkt faustregel '
           ':........................................................................................................................................................................................................',
 '100126': "definiere faustregeln in der form ' es i

In [ ]:
# then use get_sent_dep_txt to filter relations in the generated RoT as ifi they were single sentences
rot_dep = {}
for id,txt in rot_sentences.items():
    print("analyzing sentence:", txt)
    try:
        relation, sent, dep = get_sent_dep_txt(txt)
        rot_dep[id] = {"rel": relation, "sent": sent, "dep": dep}
    except:
        print("error analyzing sentence:", txt)
print(rot_dep)